# 2. Window features and master tables

1. **Window features**, one experiment at a time: 60-frame windows with a 30-frame overlap, about 30 motion
   features per window (speed, MSD fit, straightness, self-intersections, ...). This is slow, so run it on the
   cluster (`hpc/README.md`) or from a terminal:

       python scripts/compute_windows.py --input_file <experiment>_analyze/saved_data/instant_df.csv --n_jobs 16

2. **Master tables**: merge all experiment folders into one frame table and one window table, with the
   metadata parsed from filenames and folder names. Which folders, and any fixes: `configs/master_tables.yaml`.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import sptnano` works from this folder

import polars as pl

from sptnano import config

paths = config.load_paths()
settings = config.load_settings()

from datetime import date
from pathlib import Path

from sptnano import master_tables, windows

## 1. Window features (one experiment)

Writes `instant_df_processed.csv` and `time_windowed_df_processed.csv` next to `instant_df.csv`.

In [ ]:
RUN_WINDOWS = False  # True = compute here instead of on the cluster

if RUN_WINDOWS:
    input_file = paths["cropped_dir"] / "saved_data" / "instant_df.csv"
    windows.compute_windows(
        input_file,
        input_file.parent,
        window_params=windows.window_params_from_settings(settings),
        frame_interval_s=settings["acquisition"]["frame_interval_s"],
        n_jobs=-1,
        chunk_size=settings["windows"]["chunk_size"],
    )

## 2. Load every folder

Reading big CSVs from the network drive (F:) is very slow and polars can hang on it.
Copy the folders to a local drive first if this takes more than a few minutes.

In [ ]:
rules = config.load_yaml("master_tables.yaml")
folders = list(rules["folders"])
print(rules["master_dir"])
print("\n".join(folders))

instant_by_folder, windowed_by_folder = master_tables.load_folder_tables(rules["master_dir"], folders)

## 3. Metadata and checks

Fixes the known wrong conditions/filenames, parses the metadata, and prints what each folder contributes.
Read the issues before merging.

In [ ]:
instant_by_folder, windowed_by_folder, lookup_by_folder = master_tables.annotate_all_folders(
    instant_by_folder, windowed_by_folder, rules
)
issues = {
    folder: master_tables.check_folder(
        folder, lookup_by_folder[folder], instant_by_folder.get(folder), windowed_by_folder.get(folder)
    )
    for folder in folders
}
master_tables.schema_differences(instant_by_folder, "instant")
master_tables.schema_differences(windowed_by_folder, "windowed")

## 4. Merge, rebuild IDs, verify, save

`unique_id` and `window_uid` are rebuilt from folder + condition + filename + particle + replicate, so they
stay unique across folders. `verify_master_df` raises an error if anything is off.

In [ ]:
def merge(dfs_by_folder):
    df = master_tables.merge_folders(dfs_by_folder, folders)
    df = master_tables.normalise_genotype_names(df)
    df = master_tables.rebuild_ids(df)
    return master_tables.fix_location_from_loc_underscore(df)


master_instant_df = merge(instant_by_folder)
master_windowed_df = merge(windowed_by_folder)

master_tables.verify_master_df(master_instant_df, "instant", folders)
master_tables.verify_master_df(master_windowed_df, "windowed", folders, expect_unique_window_rows=True)
master_tables.verify_instant_windowed_window_uid_mapping(master_instant_df, master_windowed_df)

In [ ]:
save_dir = Path(rules["save_dir"])
if (save_dir / "master_instant_df.parquet").exists():
    save_dir = save_dir.with_name(f"{save_dir.name}_{date.today():%Y%m%d}")  # never overwrite existing master tables
master_tables.save_master_tables(master_instant_df, master_windowed_df, save_dir, lookup_by_folder)